# 🚀 Notebook 2: Database Optimization

Before adding caching or replicas, optimize your existing database. Proper indexing solves most read scaling problems.

## Learning Objectives

By the end of this notebook, you'll understand:
- How indexes work (B-tree, Hash)
- Creating effective indexes
- Composite indexes and column order
- Reading EXPLAIN output

---

🔍 **Open Adminer** at http://localhost:8080 to run queries and see execution plans!

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [1]:
import psycopg2
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def run_query(query: str):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    try:
        results = cursor.fetchall()
    except:
        results = []
    conn.commit()
    conn.close()
    return results

def measure_query(query: str) -> tuple:
    conn = get_connection()
    cursor = conn.cursor()
    start = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()
    return elapsed, len(results)

def explain_query(query: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(f"EXPLAIN ANALYZE {query}")
    plan = [row[0] for row in cursor.fetchall()]
    conn.close()
    return plan

print("✅ Connected to PostgreSQL")

✅ Connected to PostgreSQL


## 📚 How Indexes Work

An index is like a book's index - instead of reading every page, you look up where to find what you need.

In [2]:
print("📚 Index Types")
print("=" * 60)
print("""
B-TREE INDEX (Default)
─────────────────────────────────────────────────────────────
• Best for: Range queries, equality, sorting
• Supports: <, <=, =, >=, >, BETWEEN, LIKE 'prefix%'
• Structure: Balanced tree with O(log n) lookups

              [50]
             /    \\
         [25]      [75]
        /    \\    /    \\
     [10]  [30] [60]  [90]

Example: Finding email = 'user50@example.com'
         Only checks ~3-4 nodes instead of 100 rows!

─────────────────────────────────────────────────────────────

HASH INDEX
─────────────────────────────────────────────────────────────
• Best for: Exact equality only (=)
• Does NOT support: Range queries, sorting
• Structure: Hash table with O(1) lookups

hash('user50@example.com') → bucket[42] → row pointer

─────────────────────────────────────────────────────────────
""")

print("💡 Use B-tree (default) unless you ONLY do exact matches!")

📚 Index Types

B-TREE INDEX (Default)
─────────────────────────────────────────────────────────────
• Best for: Range queries, equality, sorting
• Supports: <, <=, =, >=, >, BETWEEN, LIKE 'prefix%'
• Structure: Balanced tree with O(log n) lookups

              [50]
             /    \
         [25]      [75]
        /    \    /    \
     [10]  [30] [60]  [90]

Example: Finding email = 'user50@example.com'
         Only checks ~3-4 nodes instead of 100 rows!

─────────────────────────────────────────────────────────────

HASH INDEX
─────────────────────────────────────────────────────────────
• Best for: Exact equality only (=)
• Does NOT support: Range queries, sorting
• Structure: Hash table with O(1) lookups

hash('user50@example.com') → bucket[42] → row pointer

─────────────────────────────────────────────────────────────

💡 Use B-tree (default) unless you ONLY do exact matches!


## 🔬 Before and After: Index Impact

In [3]:
run_query("DROP INDEX IF EXISTS idx_users_email")

print("🔬 BEFORE INDEX: Find user by email")
print("=" * 60)

query = "SELECT * FROM users WHERE email = 'user50@example.com'"

elapsed, count = measure_query(query)
print(f"\nTime: {elapsed:.2f}ms | Rows: {count}")
print("\nExecution Plan:")
for line in explain_query(query):
    print(f"  {line}")

🔬 BEFORE INDEX: Find user by email

Time: 1.14ms | Rows: 1

Execution Plan:
  Index Scan using idx_users_email_covering on users  (cost=0.29..8.30 rows=1 width=606) (actual time=0.023..0.024 rows=1 loops=1)
    Index Cond: ((email)::text = 'user50@example.com'::text)
  Planning Time: 0.226 ms
  Execution Time: 0.043 ms


In [4]:
print("🔨 Creating index on users.email...")
run_query("CREATE INDEX idx_users_email ON users(email)")
print("✅ Index created!\n")

print("🔬 AFTER INDEX: Find user by email")
print("=" * 60)

elapsed, count = measure_query(query)
print(f"\nTime: {elapsed:.2f}ms | Rows: {count}")
print("\nExecution Plan:")
for line in explain_query(query):
    print(f"  {line}")

print("\n💡 Notice 'Index Scan' instead of 'Seq Scan'!")

🔨 Creating index on users.email...
✅ Index created!

🔬 AFTER INDEX: Find user by email

Time: 0.78ms | Rows: 1

Execution Plan:
  Index Scan using idx_users_email on users  (cost=0.29..8.30 rows=1 width=606) (actual time=0.009..0.009 rows=1 loops=1)
    Index Cond: ((email)::text = 'user50@example.com'::text)
  Planning Time: 0.126 ms
  Execution Time: 0.020 ms

💡 Notice 'Index Scan' instead of 'Seq Scan'!


## 🎯 Composite Indexes

When queries filter on multiple columns, composite indexes help.

In [5]:
print("🎯 Composite Index: Products by category and price")
print("=" * 60)

query = """
SELECT * FROM products 
WHERE category = 'Electronics' AND price < 100
ORDER BY price
"""

run_query("DROP INDEX IF EXISTS idx_products_category_price")

print("\nBEFORE composite index:")
elapsed, count = measure_query(query)
print(f"Time: {elapsed:.2f}ms | Rows: {count}")
for line in explain_query(query)[:3]:
    print(f"  {line}")

🎯 Composite Index: Products by category and price

BEFORE composite index:
Time: 1.44ms | Rows: 167
  Sort  (cost=170.43..170.86 rows=173 width=109) (actual time=0.248..0.251 rows=167 loops=1)
    Sort Key: price
    Sort Method: quicksort  Memory: 48kB


In [6]:
print("\n🔨 Creating composite index (category, price)...")
run_query("CREATE INDEX idx_products_category_price ON products(category, price)")

print("\nAFTER composite index:")
elapsed, count = measure_query(query)
print(f"Time: {elapsed:.2f}ms | Rows: {count}")
for line in explain_query(query)[:3]:
    print(f"  {line}")


🔨 Creating composite index (category, price)...

AFTER composite index:


Time: 1.29ms | Rows: 167


  Sort  (cost=104.57..105.00 rows=173 width=109) (actual time=0.081..0.085 rows=167 loops=1)
    Sort Key: price
    Sort Method: quicksort  Memory: 48kB


## ⚠️ Column Order Matters!

In [7]:
print("⚠️ Composite Index Column Order")
print("=" * 60)
print("""
Index on (category, price) supports:
─────────────────────────────────────────────────────────────
✅ WHERE category = 'X'                    (leftmost column)
✅ WHERE category = 'X' AND price < 100    (both columns)
❌ WHERE price < 100                       (skips leftmost!)

Think of it like a phone book:
─────────────────────────────────────────────────────────────
• Sorted by (LastName, FirstName)
• Easy to find all "Smith"s
• Easy to find "Smith, John"
• Hard to find all "John"s (scattered throughout!)
""")

print("\n🔬 Query using ONLY price (skips category):")
query_price_only = "SELECT * FROM products WHERE price < 50"
for line in explain_query(query_price_only)[:2]:
    print(f"  {line}")
print("\n💡 Falls back to Seq Scan because price isn't leftmost!")

⚠️ Composite Index Column Order

Index on (category, price) supports:
─────────────────────────────────────────────────────────────
✅ WHERE category = 'X'                    (leftmost column)
✅ WHERE category = 'X' AND price < 100    (both columns)
❌ WHERE price < 100                       (skips leftmost!)

Think of it like a phone book:
─────────────────────────────────────────────────────────────
• Sorted by (LastName, FirstName)
• Easy to find all "Smith"s
• Easy to find "Smith, John"
• Hard to find all "John"s (scattered throughout!)


🔬 Query using ONLY price (skips category):
  Seq Scan on products  (cost=0.00..151.50 rows=390 width=109) (actual time=0.002..0.216 rows=380 loops=1)
    Filter: (price < '50'::numeric)

💡 Falls back to Seq Scan because price isn't leftmost!


## 📊 Index Recommendations

In [8]:
print("📊 When to Create Indexes")
print("=" * 60)
print("""
✅ DO INDEX:
─────────────────────────────────────────────────────────────
• Primary keys (automatic)
• Foreign keys used in JOINs
• Columns in WHERE clauses
• Columns in ORDER BY
• Columns with high cardinality (many unique values)

❌ DON'T INDEX:
─────────────────────────────────────────────────────────────
• Tiny tables (< 1000 rows)
• Columns with low cardinality (e.g., boolean, status)
• Columns rarely used in queries
• Tables with heavy writes and few reads

⚖️ TRADE-OFFS:
─────────────────────────────────────────────────────────────
• Indexes speed up reads but slow down writes
• Each index adds storage overhead
• Too many indexes → slower INSERT/UPDATE/DELETE
• For read-heavy apps, index liberally!
""")

📊 When to Create Indexes

✅ DO INDEX:
─────────────────────────────────────────────────────────────
• Primary keys (automatic)
• Foreign keys used in JOINs
• Columns in WHERE clauses
• Columns in ORDER BY
• Columns with high cardinality (many unique values)

❌ DON'T INDEX:
─────────────────────────────────────────────────────────────
• Tiny tables (< 1000 rows)
• Columns with low cardinality (e.g., boolean, status)
• Columns rarely used in queries
• Tables with heavy writes and few reads

⚖️ TRADE-OFFS:
─────────────────────────────────────────────────────────────
• Indexes speed up reads but slow down writes
• Each index adds storage overhead
• Too many indexes → slower INSERT/UPDATE/DELETE
• For read-heavy apps, index liberally!



In [9]:
print("📋 Creating Common Indexes for Our Schema")
print("=" * 60)

indexes = [
    ("idx_posts_user_id", "posts(user_id)", "Find posts by author"),
    ("idx_posts_created_at", "posts(created_at DESC)", "Recent posts"),
    ("idx_comments_post_id", "comments(post_id)", "Comments on a post"),
    ("idx_reviews_product_id", "reviews(product_id)", "Reviews for product"),
    ("idx_short_urls_code", "short_urls(short_code)", "URL lookup"),
]

for idx_name, idx_def, description in indexes:
    run_query(f"DROP INDEX IF EXISTS {idx_name}")
    run_query(f"CREATE INDEX {idx_name} ON {idx_def}")
    print(f"✅ {idx_name}: {description}")

print("\n💡 These indexes will help all our subsequent queries!")

📋 Creating Common Indexes for Our Schema


✅ idx_posts_user_id: Find posts by author
✅ idx_posts_created_at: Recent posts


✅ idx_comments_post_id: Comments on a post
✅ idx_reviews_product_id: Reviews for product


✅ idx_short_urls_code: URL lookup

💡 These indexes will help all our subsequent queries!


## 🔍 Query Optimization Tips

In [10]:
print("🔍 Query Optimization Tips")
print("=" * 60)
print("""
1. SELECT ONLY WHAT YOU NEED
─────────────────────────────────────────────────────────────
❌ SELECT * FROM users WHERE id = 1
✅ SELECT username, email FROM users WHERE id = 1

2. USE LIMIT FOR PAGINATION
─────────────────────────────────────────────────────────────
❌ SELECT * FROM posts ORDER BY created_at DESC
✅ SELECT * FROM posts ORDER BY created_at DESC LIMIT 20

3. AVOID FUNCTIONS ON INDEXED COLUMNS
─────────────────────────────────────────────────────────────
❌ WHERE LOWER(email) = 'user@example.com'  -- Can't use index
✅ WHERE email = 'user@example.com'          -- Uses index

4. USE EXISTS INSTEAD OF COUNT FOR EXISTENCE CHECKS
─────────────────────────────────────────────────────────────
❌ SELECT COUNT(*) FROM likes WHERE post_id = 1  -- Scans all
✅ SELECT EXISTS(SELECT 1 FROM likes WHERE post_id = 1)  -- Stops early

5. BATCH QUERIES WHEN POSSIBLE
─────────────────────────────────────────────────────────────
❌ Loop: SELECT * FROM users WHERE id = 1, 2, 3...
✅ SELECT * FROM users WHERE id IN (1, 2, 3, ...)
""")

🔍 Query Optimization Tips

1. SELECT ONLY WHAT YOU NEED
─────────────────────────────────────────────────────────────
❌ SELECT * FROM users WHERE id = 1
✅ SELECT username, email FROM users WHERE id = 1

2. USE LIMIT FOR PAGINATION
─────────────────────────────────────────────────────────────
❌ SELECT * FROM posts ORDER BY created_at DESC
✅ SELECT * FROM posts ORDER BY created_at DESC LIMIT 20

3. AVOID FUNCTIONS ON INDEXED COLUMNS
─────────────────────────────────────────────────────────────
❌ WHERE LOWER(email) = 'user@example.com'  -- Can't use index
✅ WHERE email = 'user@example.com'          -- Uses index

4. USE EXISTS INSTEAD OF COUNT FOR EXISTENCE CHECKS
─────────────────────────────────────────────────────────────
❌ SELECT COUNT(*) FROM likes WHERE post_id = 1  -- Scans all
✅ SELECT EXISTS(SELECT 1 FROM likes WHERE post_id = 1)  -- Stops early

5. BATCH QUERIES WHEN POSSIBLE
─────────────────────────────────────────────────────────────
❌ Loop: SELECT * FROM users WHERE id = 1, 

## 🧪 Quick Quiz

1. **What's the difference between B-tree and Hash indexes?**

2. **For index (A, B, C), which queries can use it?**

3. **Why not index every column?**

## 🎯 Covering Indexes (PostgreSQL `INCLUDE`)

A **covering index** stores extra columns inside the index itself so the
database can answer the whole query from the index — without touching the
table. This is sometimes called an "index-only scan".

```
Normal index on (email):
   lookup email → get row pointer → fetch row from table  (2 steps)

Covering index on (email) INCLUDE (username, display_name):
   lookup email → index already has username + display_name  (1 step)
```

**When to use**: hot read paths where you always select the same few columns.
Classic example: authentication (look up user by email, only need id + password_hash).


In [11]:
# Demo: covering index on users(email) INCLUDE (username, display_name)

query = "SELECT username, display_name FROM users WHERE email = 'user500@example.com'"

run_query("DROP INDEX IF EXISTS idx_users_email")
run_query("DROP INDEX IF EXISTS idx_users_email_covering")

# Plain index
run_query("CREATE INDEX idx_users_email ON users(email)")
print("Plain index on (email):")
for line in explain_query(query)[:4]:
    print(f"  {line}")
elapsed, _ = measure_query(query)
print(f"  Time: {elapsed:.2f}ms\n")

# Covering index — the engine can answer from the index alone
run_query("DROP INDEX idx_users_email")
run_query(
    "CREATE INDEX idx_users_email_covering "
    "ON users(email) INCLUDE (username, display_name)"
)
print("Covering index on (email) INCLUDE (username, display_name):")
for line in explain_query(query)[:4]:
    print(f"  {line}")
elapsed, _ = measure_query(query)
print(f"  Time: {elapsed:.2f}ms")
print("\n💡 Look for 'Index Only Scan' — that means zero table reads!")


Plain index on (email):
  Index Scan using idx_users_email on users  (cost=0.29..8.30 rows=1 width=19) (actual time=0.015..0.015 rows=1 loops=1)
    Index Cond: ((email)::text = 'user500@example.com'::text)
  Planning Time: 0.125 ms
  Execution Time: 0.027 ms


  Time: 0.69ms

Covering index on (email) INCLUDE (username, display_name):
  Index Only Scan using idx_users_email_covering on users  (cost=0.29..8.30 rows=1 width=19) (actual time=0.015..0.015 rows=1 loops=1)
    Index Cond: (email = 'user500@example.com'::text)
    Heap Fetches: 0
  Planning Time: 0.129 ms


  Time: 0.70ms

💡 Look for 'Index Only Scan' — that means zero table reads!


## 🎯 Partial Indexes (index a SUBSET of rows)

If most queries only care about a small slice of the table, a **partial index**
skips the rest — smaller index, faster updates, cheaper storage.

```sql
-- Only index posts with lots of engagement (the "hot" ones)
CREATE INDEX idx_viral_posts
    ON posts(created_at DESC)
    WHERE like_count > 500;
```

**Real-world examples**:
- `WHERE deleted_at IS NULL` — only index active rows
- `WHERE status = 'pending'` — only index rows the worker queries
- `WHERE is_public = true` — only index publicly visible content


In [12]:
# Demo: partial index for "viral" posts only
run_query("DROP INDEX IF EXISTS idx_viral_posts")
run_query(
    "CREATE INDEX idx_viral_posts ON posts(created_at DESC) "
    "WHERE like_count > 500"
)

# Check index size — partial index is MUCH smaller than a full index
size = run_query(
    "SELECT pg_size_pretty(pg_relation_size('idx_viral_posts'))"
)
print(f"Partial index size: {size[0][0]}")

query = (
    "SELECT id, content, like_count FROM posts "
    "WHERE like_count > 500 ORDER BY created_at DESC LIMIT 10"
)
print("\nExecution plan (should use idx_viral_posts):")
for line in explain_query(query)[:4]:
    print(f"  {line}")


Partial index size: 560 kB

Execution plan (should use idx_viral_posts):


  Limit  (cost=0.29..1.70 rows=10 width=73) (actual time=0.022..0.040 rows=10 loops=1)
    ->  Index Scan using idx_viral_posts on posts  (cost=0.29..3511.55 rows=24778 width=73) (actual time=0.022..0.039 rows=10 loops=1)
  Planning Time: 0.162 ms
  Execution Time: 0.046 ms


In [13]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. B-tree vs Hash:")
print("   B-tree: Range queries, sorting, equality")
print("   Hash: Only equality (=), faster for exact match")
print("   Default to B-tree unless you know otherwise")
print()
print("2. Index (A, B, C) supports:")
print("   ✅ WHERE A = x")
print("   ✅ WHERE A = x AND B = y")
print("   ✅ WHERE A = x AND B = y AND C = z")
print("   ❌ WHERE B = y (skips A)")
print("   ❌ WHERE C = z (skips A, B)")
print()
print("3. Why not index everything:")
print("   - Slows down INSERT/UPDATE/DELETE")
print("   - Uses disk space")
print("   - Index maintenance overhead")
print("   - For read-heavy apps, still index liberally!")

📝 Quiz Answers

1. B-tree vs Hash:
   B-tree: Range queries, sorting, equality
   Hash: Only equality (=), faster for exact match
   Default to B-tree unless you know otherwise

2. Index (A, B, C) supports:
   ✅ WHERE A = x
   ✅ WHERE A = x AND B = y
   ✅ WHERE A = x AND B = y AND C = z
   ❌ WHERE B = y (skips A)
   ❌ WHERE C = z (skips A, B)

3. Why not index everything:
   - Slows down INSERT/UPDATE/DELETE
   - Uses disk space
   - Index maintenance overhead
   - For read-heavy apps, still index liberally!


## 📚 Summary

### Key Takeaways

1. **Indexes turn O(n) into O(log n)** - massive speedup
2. **Use EXPLAIN** - see how queries actually execute
3. **Column order matters** - leftmost columns used first
4. **Index for your queries** - not generic "best practices"
5. **B-tree is usually right** - handles most use cases

### Next Up

In **Notebook 3**, we'll learn about denormalization:
- Trading storage for speed
- Materialized views
- Pre-computed aggregations